# Localized MCQ/chat consistency — primary models

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        q=p/"tasks/political-compass/qualitative-analysis"
        if (q/"config.py").exists(): return q
        if p.name=="qualitative-analysis" and (p/"config.py").exists(): return p
    raise FileNotFoundError
ROOT=find_root(); TABLES=ROOT/"artifacts/tables"; FIGURES=ROOT/"artifacts/figures"
CHAT_ORDER=["gemma-3-1b-it","gemma-3-4b-it","gemma-3-12b-it","gemma-3-27b-it",
"Qwen3-4B_no_think","Qwen3-8B_no_think","Qwen3-14B_no_think","Qwen3-32B_no_think",
"Qwen3-4B_think","Qwen3-8B_think","Qwen3-14B_think","Qwen3-32B_think"]
LABEL=dict(zip(CHAT_ORDER,["Gemma 1B","Gemma 4B","Gemma 12B","Gemma 27B",
"Qwen 4B","Qwen 8B","Qwen 14B","Qwen 32B","Qwen 4B Think","Qwen 8B Think",
"Qwen 14B Think","Qwen 32B Think"]))
def ordered(d,col="model_variant"):
    d=d[d[col].isin(CHAT_ORDER)].copy(); d["model_label"]=d[col].map(LABEL)
    d["model_label"]=pd.Categorical(d.model_label,[LABEL[x] for x in CHAT_ORDER],ordered=True)
    return d.sort_values("model_label")
sns.set_theme(style="whitegrid")


Each base-model × method × question cell has only 12 exact overlapping prompt configurations. These are candidates for replication, not stable model effects. Ordering follows Gemma sizes, then Qwen sizes, with standard chat before think chat.

In [2]:
p=pd.read_parquet(TABLES/"mcq_crosscheck/matched_question_pairs.parquet")
g=p.groupby(["method","base_model","question_id"]).agg(n=("four_way_agree","size"),four_way=("four_way_agree","mean"),binary=("binary_stance_agree","mean"),mean_tv=("total_variation","mean"),answer_shift=("expected_answer_delta","mean")).reset_index()
gem=["gemma-3-1b-it","gemma-3-4b-it","gemma-3-12b-it","gemma-3-27b-it"]; qw=["Qwen3-4B","Qwen3-8B","Qwen3-14B","Qwen3-32B"]
order=[(x,"Standard Chat") for x in gem]+sum(([ (x,"Standard Chat"),(x,"Chat Think") ] for x in qw),[])
g["order_key"]=[order.index((b,m)) if (b,m) in order else 999 for b,m in zip(g.base_model,g.method)]; g=g[g.order_key<999]
display(g.sort_values("mean_tv",ascending=False).head(60))
r=g.assign(high=g.binary>=.75,low=g.binary<=.50).groupby(["method","question_id"]).agg(models=("base_model","nunique"),high_models=("high","sum"),low_models=("low","sum"),median_tv=("mean_tv","median")).reset_index()
display(r.sort_values(["low_models","median_tv"],ascending=False).head(40))

,method,base_model,question_id,n,four_way,binary,mean_tv,answer_shift,order_key
722,Standard Chat,gemma-3-4b-it,40,12,0.083333,0.333333,0.945092,-0.126571,1
723,Standard Chat,gemma-3-4b-it,41,12,0.083333,0.583333,0.937267,0.099003,1
719,Standard Chat,gemma-3-4b-it,37,12,0.083333,0.583333,0.926339,0.013376,1
733,Standard Chat,gemma-3-4b-it,51,12,0.083333,0.583333,0.920170,0.173581,1
716,Standard Chat,gemma-3-4b-it,34,12,0.166667,0.250000,0.885859,0.522682,1
687,Standard Chat,gemma-3-4b-it,5,12,0.000000,0.250000,0.883443,0.918567,1
590,Standard Chat,gemma-3-1b-it,32,12,0.083333,0.500000,0.882800,0.746268,0
586,Standard Chat,gemma-3-1b-it,28,12,0.000000,0.333333,0.875762,0.531837,0
609,Standard Chat,gemma-3-1b-it,51,12,0.083333,0.333333,0.873566,0.388911,0
606,Standard Chat,gemma-3-1b-it,48,12,0.000000,0.333333,0.872531,0.471405,0


,method,question_id,models,high_models,low_models,median_tv
4,Chat Think,4,4,0,4,0.792258
94,Standard Chat,32,8,2,4,0.685785
85,Standard Chat,23,8,1,4,0.653607
89,Standard Chat,27,8,1,4,0.628235
97,Standard Chat,35,8,2,4,0.624795
123,Standard Chat,61,8,2,4,0.554816
38,Chat Think,38,4,0,3,0.720364
32,Chat Think,32,4,1,3,0.709376
42,Chat Think,42,4,1,3,0.708064
113,Standard Chat,51,8,4,3,0.674052
